# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,IREN,47,1,-0.707500,2026-07-19 08:14:29+00:00
1,SPCX,19,8,-0.127163,2026-07-22 19:47:53+00:00
2,SPY,6,5,0.241100,2026-07-21 20:21:02+00:00
3,NVDA,5,4,0.066750,2026-07-22 20:43:23+00:00
4,META,4,4,0.100625,2026-07-22 17:18:39+00:00
5,DRAM,7,3,-0.349167,2026-07-21 13:12:44+00:00
6,MSFT,4,3,0.689867,2026-07-22 17:18:39+00:00
7,GOOGL,4,3,0.614133,2026-07-22 18:26:37+00:00
8,AMD,3,3,0.490267,2026-07-22 12:53:08+00:00
9,SMCI,3,3,0.031500,2026-07-22 11:47:54+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['IREN', 'SPCX', 'SPY', 'NVDA']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    col_data = in_df[column_name].dropna()
    if col_data.empty:
        return None
    return caster(col_data.iloc[-1])


price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": safe_latest_value(features_df, "Close", float),
        "latest_rsi": safe_latest_value(features_df, "RSI", float),
        "latest_macd": safe_latest_value(features_df, "MACD", float),
        "return_21d": safe_latest_value(features_df, "21D Return", float),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,NVDA,251,212.059998,56.182705,0.733478,0.016343,5.406238,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== IREN latest metrics =====
                Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                    
2026-07-16  34.830002  41339100            8.808991           49.729667            48.802995  31.917762 -4.727280  -0.165149   -0.427609
2026-07-17  33.619999  47227400            8.765065           48.667666            48.455086  30.727644 -4.980302  -0.182790   -0.431903
2026-07-20  40.200001  93338200            8.529846           47.945667            48.265939  43.143156 -4.596882   0.031298   -0.308209
2026-07-21  41.290001  51030000            8.524987           47.510333            48.106113  44.904743 -4.157145   0.069966   -0.311374
2026-07-22  41.279999  40847104            8.303088           46.913333            47.949730  44.890998 -3.766044   0.078370   -0.274134

===== S

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== IREN regression results =====
                Close  Predicted Close
Date                                  
2026-07-09  41.720001        43.007701
2026-07-10  41.139999        43.077951
2026-07-13  38.980000        39.833321
2026-07-14  38.590000        39.503156
2026-07-15  38.279999        37.890071
2026-07-16  34.830002        34.660476
2026-07-17  33.619999        32.769701
2026-07-20  40.200001        38.823622
2026-07-21  41.290001        40.736379
2026-07-22  41.279999        41.103217

===== SPCX regression results =====
Not enough data for regression output.

===== SPY regression results =====
                 Close  Predicted Close
Date                                   
2026-07-09  751.710022       753.207011
2026-07-10  754.950012       756.805653
2026-07-13  749.169983       752.635881
2026-07-14  751.830017       754.420206
2026-07-15  754.809998       757.018426
2026-07-16  750.719971       751.646243
2026-07-17  743.289978       745.165909
2026-07-20  742.090027 

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "IREN",
    "SPCX",
    "SPY",
    "NVDA"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
